## Ensembles: Random Forest, Gradient Boosting en XGBoost

**Wat is een ensemble?**
Een ensemble combineert de voorspellingen van meerdere modellen tot één voorspelling. Eén decision tree maakt snel fouten en overfit makkelijk, maar als je veel verschillende bomen combineert, middelen de fouten elkaar uit.

Er zijn twee hoofdtechnieken:
- **Bagging** (Random Forest): veel bomen worden onafhankelijk van elkaar getraind, elk op een willekeurige steekproef van de data (bootstrap) en met een willekeurige selectie van features. De voorspelling is de meerderheid van de stemmen. Dit verlaagt vooral de **variance**.
- **Boosting** (Gradient Boosting, XGBoost): bomen worden na elkaar getraind. Elke nieuwe boom probeert de fouten van de vorige bomen te corrigeren. Dit verlaagt vooral de **bias**.

**Waarom ensembles voor deze dataset?**
Decision trees kunnen niet-lineaire verbanden en interacties oppikken, zoals het effect van een hoge `avg_glucose_level` of de combinatie van hoge leeftijd en hoge bloeddruk. Logistic regression kan dit niet.

Ensembles op basis van decision trees hebben geen standaardisatie nodig: een boom splitst op één feature tegelijk, waardoor de schaal geen invloed heeft.

### Random Forest: baseline

We beginnen met een Random Forest met standaardinstellingen en 300 bomen. We evalueren op dezelfde manier als bij logistic regression: 5-fold `StratifiedKFold`.

In [1]:
"""Ensembles voor de Kaggle-competitie Predicting stroke."""
import sys

sys.path.append("..")  # zodat de map src gevonden wordt

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, precision_score
from sklearn.model_selection import StratifiedKFold, cross_validate

from src.preprocessing import StrokeData

data = StrokeData()
X, y = data.get_xy()
X_test = data.get_test()

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": "recall",
    "f1": "f1",
}

rf_default = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)

scores = cross_validate(rf_default, X, y, cv=cv, scoring=scoring)
for metric in scoring:
    print(f"{metric}: {scores[f'test_{metric}'].mean():.3f}")

precision: 0.000
recall: 0.000
f1: 0.000


**Observatie:** net als de baseline van logistic regression voorspelt het Random Forest voor geen enkele patiënt een herseninfarct. Bijna alle bladeren van de bomen bevatten vooral patiënten zonder herseninfarct, waardoor de meerderheid van de 300 bomen altijd "geen herseninfarct" stemt. Een krachtiger model lost de onbalans dus niet vanzelf op.

### Class weights

Bij logistic regression loste `class_weight="balanced"` het probleem direct op. We proberen hetzelfde bij het Random Forest: een patiënt met een herseninfarct telt dan zwaarder mee bij het kiezen van de splits en bij het stemmen in de bladeren.

In [2]:
rf_balanced = RandomForestClassifier(
    n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1
)

scores = cross_validate(rf_balanced, X, y, cv=cv, scoring=scoring)
for metric in scoring:
    print(f"{metric}: {scores[f'test_{metric}'].mean():.3f}")

precision: 0.080
recall: 0.021
f1: 0.033


**Observatie:** anders dan bij logistic regression helpt `class_weight="balanced"` hier nauwelijks: de recall is slechts 0,021. De oorzaak is overfitting. Standaard groeit elke boom door tot alle bladeren zuiver zijn (alleen patiënten van één klasse). De bomen leren de trainingsdata dan uit het hoofd. In een zuiver blad valt er niets te wegen, dus de class weights hebben geen effect. Nieuwe patiënten met een herseninfarct komen vrijwel altijd terecht in een blad met "geen herseninfarct".